# Understanding LEDGAR  — NVIDIA GPU copy

> **GPU-ready copy.** Originals live in the parent folder and are untouched.
> Pure data walkthrough (no training) — useful on the cluster so the team shares one view of the data.
> Detects `cuda` automatically. Run the install cell first on a fresh cluster node.

In [ ]:
# --- Run once on the NVIDIA cluster to install dependencies ---
# NOTE: on the TR H100 cluster, install torch with the correct CUDA wheel FIRST, e.g.:
#   pip install torch --index-url https://download.pytorch.org/whl/cu124
# then the rest:
!pip install -q "transformers>=4.44" "datasets>=2.19,<3" scikit-learn accelerate
print("deps installed")

# Understanding LEDGAR — a beginner's dataset walkthrough

**Same spirit as a good first look at any dataset:** we go slow, we **don't train
anything**, we just *open the data and look* until it's obvious.

**What LEDGAR is, and why it's a great dataset to learn fine-tuning on:** LEDGAR is a
**classification** task — you read one short contract clause and pick the single label that
describes it (out of 100 provision types). Classification is the *simplest* fine-tuning
shape there is: one piece of text in, one label out. No pointing to positions in the text,
no "the answer might be missing" cases — just "which category is this?" That simplicity is
exactly what makes it the right place to learn the training mechanics clearly.

By the end you'll understand:
1. What LEDGAR contains and what the task shape is (classification).
2. What one example looks like (`text` + `label`).
3. What the 100 labels are, and that they're **imbalanced** (this matters for training).
4. How long the texts are (why this fits small models easily).

> **Kernel:** set it (top-right) to **`cuad-finetune (.venv)`** — that's just the name of the
> Python environment we set up; it has all the libraries this notebook needs.

## Part 0 — What "classification" means here (the one idea to hold onto)

There are a few different "shapes" a text task can take. LEDGAR is the **classification**
shape, which is the easiest to reason about:

| | **Classification (LEDGAR)** |
|---|---|
| Input | **One short clause** of contract text |
| Output | **One label** — pick from a fixed list of 100 |
| Analogy | "Put this document in the right folder" |
| What the model produces | one score per possible label; the highest score wins |
| Difficulty | **Low — ideal for learning the training loop** |

**LEDGAR's job in plain words:** *"Here is a paragraph from a contract. Which of 100
provision types is it?"* e.g. this paragraph is a *Governing Laws* clause; that one is a
*Termination* clause; this one is *Confidentiality*. That's the entire task.

> For contrast, other tasks ask the model to *find and highlight a span* inside a long
> document, or to *generate* free text. Those are harder to set up and grade. LEDGAR
> deliberately avoids all of that — which is why we start here.

Let's load it and see.

## Part 1 — Load LEDGAR and see its shape

LEDGAR lives inside the **LexGLUE** benchmark (a suite of legal NLP tasks). We ask for the
`"ledgar"` configuration specifically. It downloads once, then caches.

In [ ]:
from datasets import load_dataset

ds = load_dataset("coastalcph/lex_glue", "ledgar")
ds   # print the structure

**What you're seeing:** three separate piles of data, each serving a different purpose:

- **train** (60,000) — the model learns from these.
- **validation** (10,000) — a *tuning* set: check progress during training without touching test.
- **test** (10,000) — the final, hidden exam we grade on at the very end.

And only **two columns**: `text` (the clause) and `label` (which of the 100 types it is).
Two columns is about as simple as a dataset gets — perfect for learning.

In [ ]:
train = ds["train"]
val   = ds["validation"]
test  = ds["test"]

print("Train rows      :", len(train))
print("Validation rows :", len(val))
print("Test rows       :", len(test))
print("Columns         :", train.column_names)

## Part 2 — What is ONE example?

Dead simple: a piece of contract text, and an integer label. Let's look at one.

In [ ]:
row = train[0]
print("TEXT (a contract provision):")
print("   ", row["text"][:400], "...")
print()
print("LABEL (as a raw integer):", row["label"])

### But what does label `97` *mean*?

The label is stored as a number for the computer, but each number maps to a human-readable
name. That mapping lives in the dataset's `features`. Let's decode it.

In [ ]:
label_feature = train.features["label"]     # this holds the number<->name mapping
label_names = label_feature.names            # the list of all 100 names

print("Total number of labels:", len(label_names))
print()
print("This example's label", row["label"], "means ->", label_feature.int2str(row["label"]))
print()
print("So this clause is a '" + label_feature.int2str(row["label"]) + "' provision.")

That's the whole example: **text in, one-of-100 label out.** No positions to locate, no
spans to highlight, no "the answer isn't here" case. This minimal shape is exactly why
LEDGAR is such a clean place to learn the training loop.

## Part 3 — The 100 provision types (the label set)

These are the 100 "folders" the model sorts clauses into. Let's list them all so you can see
the vocabulary of the task.

In [ ]:
for i, name in enumerate(label_names):
    end = "\n" if (i + 1) % 4 == 0 else "    "   # 4 per line, tidy columns
    print(f"{i:3d} {name:22s}", end=end)
print()

These are everyday contract concepts — Governing Laws, Terminations, Assignments, Waivers,
Confidentiality, Indemnifications, and so on. If you've ever skimmed a contract, most of
these names will feel familiar. The model's whole job is to read a clause and pick which of
these names fits.

## Part 4 — See real clauses for a few labels

Numbers are abstract; let's read actual clauses so the labels feel concrete. We'll grab one
example each for a few familiar provision types.

In [ ]:
wanted = ["Governing Laws", "Arbitration", "Terminations", "Confidentiality", "Waivers"]
found = {}
for text, lab in zip(train["text"], train["label"]):
    name = label_names[lab]
    if name in wanted and name not in found:
        found[name] = text
    if len(found) == len(wanted):
        break

for name in wanted:
    print(f"===== {name} =====")
    print(found[name][:280].strip(), "...")
    print()

Notice how *recognizable* each one is — a Governing Laws clause literally says "shall
be construed... in accordance with the laws of the State of...". This is why classification
works well: the label is strongly signalled by the words. The model just has to learn those
patterns.

## Part 5 — The labels are IMBALANCED (important for training!)

Here's a real-world wrinkle that will affect your fine-tuning. The 100 labels are **not**
evenly represented. Some provision types are extremely common (every contract has a
*Governing Laws* clause); others are rare. Let's measure it.

In [ ]:
import collections

counts = collections.Counter(train["label"])   # how many examples per label

top = counts.most_common(8)
bottom = counts.most_common()[-8:]

print("MOST common provision types:")
for lab, n in top:
    print(f"   {n:5d}  {label_names[lab]}")

print("\nLEAST common provision types:")
for lab, n in bottom:
    print(f"   {n:5d}  {label_names[lab]}")

print(f"\nBiggest class is ~{top[0][1] // bottom[-1][1]}x larger than the smallest!")

**Why you must know this:** if 3,167 clauses are *Governing Laws* but only 23 are
*Books*, a lazy model can get a deceptively good overall score by just being good at the
common classes and ignoring the rare ones entirely. When we train, we'll watch **per-class
metrics** (not just overall accuracy) so we can catch that kind of laziness.

This is also why people report **macro-F1** (which averages each class equally, so a rare
class counts as much as a common one) for LEDGAR, rather than plain accuracy (which the big
classes would dominate).

## Part 6 — How long are the texts? (why this fits easily)

Models read text in fixed-size windows (measured in "tokens" — word pieces). If a piece of
text is longer than the window, you'd have to split it into overlapping chunks, which adds
complexity. LEDGAR texts are single provisions, so they're short — let's confirm that a
normal window comfortably covers them, meaning **no chunking machinery is needed here.**

In [ ]:
lengths = [len(t.split()) for t in train["text"][:5000]]   # word counts on a sample
lengths.sort()

import statistics
print("Words per clause (sample of 5000):")
print("   shortest :", lengths[0])
print("   median   :", int(statistics.median(lengths)))
print("   longest  :", lengths[-1])
print("   95th pct :", lengths[int(0.95*len(lengths))])
print()
print("Most clauses fit in a few hundred tokens -> a single 512-token window covers them.")
print("=> No sliding window / chunking needed. Tokenization will be simple.")

## Recap — you now understand LEDGAR

1. **Task shape:** classification — read one clause, pick 1 of **100** provision labels.
2. **Columns:** just `text` + `label` (an integer that maps to a name).
3. **Size:** 60k train / 10k val / 10k test.
4. **Labels are imbalanced** — some ~100x rarer than others → watch per-class / macro-F1.
5. **Texts are short** — no sliding window; tokenization is simple.

### What's next
Now that the data is clear, the **next notebook** will *fine-tune* a small model to do this
classification — and because the task is simple, every training step will be easy to follow.
That's where you'll finally see `trainer.train()` do its thing on a task you fully understand.